# CityScore Data Loading And Cleaning

This notebook loads the raw Excel files, cleans the report-style inputs, and writes cleaned CSV outputs into the `cleaned csv` folder.

Cell guide:
- Cell 1 imports pandas and configures notebook display settings.
- Cell 2 defines folders, file paths, and validates that the inputs exist.
- Cell 3 loads the Excel files into pandas DataFrames and shows a quick inventory.
- Cell 4 defines reusable cleaning functions.
- Cell 5 applies the cleaning logic and exports cleaned CSV files.
- Cell 6 previews one cleaned dataset for a quick sanity check.


In [1]:
# Cell 1: import the core libraries and make pandas output easier to inspect in the notebook.
from pathlib import Path

import pandas as pd

# These display settings help us see more columns and rows during profiling and cleaning.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 140)


In [2]:
# Cell 2: define the project paths, create the output folder, and validate the expected input files.
# This keeps the notebook portable whether it is run from the project root or the Codes folder.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "Excel").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "Excel"
CLEANED_DIR = PROJECT_ROOT / "cleaned csv"
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

# Centralize the input files so the rest of the notebook can reference them consistently.
SOURCE_FILES = {
    "full_metric_list": DATA_DIR / "CityScore_Full_Metric_list.xlsx",
    "cityscore_summary": DATA_DIR / "CityScore_Summary.xlsx",
    "city_score_agg_report": DATA_DIR / "rpt_city_score_agg_v.csv.xlsx",
    "city_score_summary_report": DATA_DIR / "rpt_city_score_summary.csv.xlsx",
}


# Fail early if any source file is missing so we do not continue with partial data.
missing_files = [str(path) for path in SOURCE_FILES.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing source files: {missing_files}")

# Return the cleaned output folder path so we can verify where files will be written.
CLEANED_DIR


PosixPath('/Users/ameyasutar1gmail.com/Desktop/IIMA/python/Boston Home Assignment/cleaned csv')

In [4]:
# Cell 3: load each Excel file into pandas and keep the DataFrames in one dictionary.
# pandas.read_excel uses the openpyxl engine for .xlsx files.
raw_frames = {
    name: pd.read_excel(path)
    for name, path in SOURCE_FILES.items()
}

# Show a quick inventory so we can confirm the file-to-DataFrame mapping and dataset sizes.
pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": frame.shape[0],
            "columns": frame.shape[1],
            "source_file": SOURCE_FILES[name].name,
        }
        for name, frame in raw_frames.items()
    ]
)


,dataset,rows,columns,source_file
0,full_metric_list,115,17,CityScore_Full_Metric_list.xlsx
1,cityscore_summary,23,8,CityScore_Summary.xlsx
2,city_score_agg_report,844,6,rpt_city_score_agg_v.csv.xlsx
3,city_score_summary_report,18235,23,rpt_city_score_summary.csv.xlsx


In [5]:
# Cell 4: define reusable cleaning helpers so the same rules can be applied every day.
# The idea is to keep file-specific cleaning small by pushing common logic into helper functions.

# Standardize column names into lowercase snake_case for easier downstream joins and references.
def snake_case_columns(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    cleaned.columns = (
        cleaned.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace("%", "pct", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace(" ", "_", regex=False)
        .str.replace(r"[^a-z0-9_]+", "", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )
    return cleaned


# Trim whitespace in text fields and convert empty-looking strings into proper missing values.
def strip_text_values(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    object_columns = cleaned.select_dtypes(include=["object", "string"]).columns
    for column in object_columns:
        cleaned[column] = cleaned[column].astype("string").str.strip()
        cleaned[column] = cleaned[column].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    return cleaned


# Remove rows and columns that are completely empty because they add noise but no information.
def drop_empty_rows_and_columns(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.dropna(how="all").copy()
    cleaned = cleaned.dropna(axis=1, how="all")
    return cleaned


# Convert Excel serial dates like 42384 into true pandas timestamps.
# If a column already contains readable dates, we keep those through the fallback parser.
def convert_excel_serial_dates(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    cleaned = df.copy()
    for column in columns:
        if column not in cleaned.columns:
            continue
        numeric_values = pd.to_numeric(cleaned[column], errors="coerce")
        excel_dates = pd.to_datetime(
            numeric_values,
            unit="D",
            origin="1899-12-30",
            errors="coerce",
        )
        parsed_dates = pd.to_datetime(cleaned[column], errors="coerce")
        cleaned[column] = excel_dates.combine_first(parsed_dates)
    return cleaned


# Convert numeric-looking columns into numbers, while protecting known text fields.
def coerce_numeric_columns(df: pd.DataFrame, skip_columns: set[str] | None = None) -> pd.DataFrame:
    cleaned = df.copy()
    skip_columns = skip_columns or set()
    for column in cleaned.columns:
        if column in skip_columns:
            continue
        numeric_values = pd.to_numeric(cleaned[column], errors="coerce")
        if numeric_values.notna().sum() > 0:
            cleaned[column] = numeric_values
    return cleaned


# File-specific cleaning for the city-level aggregate history report.
def clean_city_score_agg_report(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = snake_case_columns(df)
    cleaned = strip_text_values(cleaned)
    cleaned = drop_empty_rows_and_columns(cleaned)
    cleaned = convert_excel_serial_dates(cleaned, ["etl_load_date"])
    cleaned = coerce_numeric_columns(cleaned, skip_columns={"etl_load_is_active_flag"})
    cleaned = cleaned.drop_duplicates().sort_values("etl_load_date").reset_index(drop=True)
    return cleaned


# File-specific cleaning for the metric-level historical summary report.
def clean_city_score_summary_report(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = snake_case_columns(df)
    cleaned = strip_text_values(cleaned)
    cleaned = drop_empty_rows_and_columns(cleaned)
    cleaned = convert_excel_serial_dates(cleaned, ["etl_load_date"])
    cleaned = coerce_numeric_columns(cleaned, skip_columns={"cty_scr_name"})
    cleaned["cty_scr_name"] = cleaned["cty_scr_name"].str.upper()
    cleaned = cleaned.drop_duplicates().sort_values(["etl_load_date", "cty_scr_name"]).reset_index(drop=True)
    return cleaned


In [6]:
# Cell 5: apply the cleaning functions, save the cleaned outputs as CSV files,
# and show a compact export summary with row counts, column counts, and output paths.
cleaned_frames = {
    "city_score_agg_clean": clean_city_score_agg_report(raw_frames["city_score_agg_report"]),
    "city_score_summary_clean": clean_city_score_summary_report(raw_frames["city_score_summary_report"]),
}

# Write each cleaned DataFrame into the cleaned csv folder so it can be reused outside the notebook.
output_files = {}
for name, frame in cleaned_frames.items():
    output_path = CLEANED_DIR / f"{name}.csv"
    frame.to_csv(output_path, index=False)
    output_files[name] = output_path

pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": frame.shape[0],
            "columns": frame.shape[1],
            "output_file": str(output_files[name].relative_to(PROJECT_ROOT)),
        }
        for name, frame in cleaned_frames.items()
    ]
)


,dataset,rows,columns,output_file
0,city_score_agg_clean,844,6,cleaned csv/city_score_agg_clean.csv
1,city_score_summary_clean,18235,23,cleaned csv/city_score_summary_clean.csv


In [7]:
# Cell 6: preview the cleaned metric-level report so we can do a quick sanity check.
cleaned_frames["city_score_summary_clean"].head()


,cty_scr_name,cty_scr_nbr_dy_01,cty_scr_nbr_dy_02,cty_scr_nbr_wk_01,cty_scr_nbr_wk_02,cty_scr_nbr_mo_01,cty_scr_nbr_mo_02,cty_scr_nbr_qt_01,cty_scr_nbr_qt_02,cty_scr_tgt_01,cty_scr_avg_01,cty_scr_avg_02,cty_scr_dev_01,cty_scr_dev_02,etl_load_date,etl_load_is_active_flag,cty_scr_open_data_source,cty_scr_metric_type,cty_scr_day,cty_scr_week,cty_scr_month,cty_scr_quarter,cty_scr_day_name
0,311 CALL CENTER PERFORMANCE,0.898000,663.0,0.892094,3605.0,0.942094,15957.0,0.955982,52933.0,0.95,NaN,NaN,NaN,NaN,1452816000000000,N,<NA>,<NA>,0.945263,0.939047,0.991678,1.006297,Thursday
1,BFD INCIDENTS,196.000000,NaN,196.571428,NaN,196.838710,NaN,205.630435,NaN,NaN,209.580645,210.054348,35.577872,33.008945,1452816000000000,N,<NA>,<NA>,1.250809,1.247173,1.245479,1.182039,Thursday
2,BFD RESPONSE TIME,0.851852,189.0,0.804069,1327.0,0.824322,5863.0,0.813523,18147.0,4.00,NaN,NaN,NaN,NaN,1452816000000000,N,<NA>,<NA>,4.695652,4.974696,4.852473,4.916887,Thursday
3,BPS ATTENDANCE,NaN,NaN,92.500000,NaN,90.714285,NaN,92.166666,NaN,95.00,NaN,NaN,NaN,NaN,1452816000000000,N,<NA>,<NA>,NaN,0.973684,0.954887,0.970175,Thursday
4,CITY SERVICES SATISFACTION SURVEYS,2.000000,1.0,3.727272,11.0,3.425000,80.0,3.622516,302.0,4.00,NaN,NaN,NaN,NaN,1452816000000000,N,<NA>,<NA>,0.500000,0.931818,0.856250,0.905629,Thursday
